# Solving Problems with SciPy

NumPy gives you fast arrays of numbers. **SciPy** (short for "Scientific Python") is built on top of NumPy and adds ready-made tools for the maths that scientists and engineers use every day: statistics, fitting curves, solving equations, measuring areas, and cleaning up noisy data.

In this notebook you'll use SciPy to investigate one day of readings from a weather sensor. If arrays or charts are new to you, try [[A6AFAE39-B14C-4CD8-B9CB-23771BBF07C9|NumPy]] and [[F9D25507-BA02-4876-9F0C-CD35B389BC47|Matplotlib]] first.

# Setting Up

Our weather sensor recorded the temperature **every two hours** for one day, from midnight (hour 0) to the next midnight (hour 24). That gives 13 readings, stored in two NumPy arrays.

SciPy is organized into **submodules**, one for each kind of problem. You import only the ones you need:

| Submodule | What it's for |
|---|---|
| `scipy.stats` | Statistics and probability |
| `scipy.interpolate` | Estimating values between data points |
| `scipy.optimize` | Fitting curves, finding best values, solving equations |
| `scipy.integrate` | Measuring the area under a curve |
| `scipy.signal` | Smoothing and analyzing signals |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hours = np.arange(0, 25, 2)
temps = np.array([11.8, 10.9, 10.1, 11.2, 14.1, 17.8, 20.9,
                  22.3, 21.6, 19.0, 16.1, 13.9, 12.4])

plt.figure(figsize=(10, 5))
plt.plot(hours, temps, 'o', color='steelblue', label='Sensor readings')
plt.title('One Day of Temperature Readings')
plt.xlabel('Hour of the day')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Statistics

The **`scipy.stats`** submodule answers questions about a set of numbers. `stats.describe()` works out several summary values in one go: the smallest and largest value, the **mean** (average), and the **variance** (how spread out the values are).

`stats.percentileofscore()` tells you what percentage of the values fall at or below a number you choose.

In [ ]:
from scipy import stats

summary = stats.describe(temps)
coldest, warmest = summary.minmax

print("Number of readings:", summary.nobs)
print("Coldest:", coldest, "°C")
print("Warmest:", warmest, "°C")
print("Mean:", round(summary.mean, 1), "°C")
print("Variance:", round(summary.variance, 1))

cool = stats.percentileofscore(temps, 15)
print(f"{cool:.0f}% of the readings were 15°C or cooler")

## The Bell Curve

`scipy.stats` also knows about **probability distributions**. The most famous is the **normal distribution**, the bell-shaped curve that describes many things in nature, such as heights, test scores, and daily temperatures.

Suppose the midday temperature in this town during July averages 22°C, and usually varies by about 3°C (the **standard deviation**). `stats.norm(22, 3)` builds that bell curve, and its `.cdf(x)` method gives the chance of a value *below* `x`.

In [ ]:
july = stats.norm(22, 3)

print(f"Chance of a day cooler than 20°C: {july.cdf(20):.0%}")
print(f"Chance of a day hotter than 28°C: {1 - july.cdf(28):.1%}")

x = np.linspace(12, 32, 200)
plt.figure(figsize=(10, 4))
plt.plot(x, july.pdf(x), color='steelblue', linewidth=2)
plt.fill_between(x, july.pdf(x), where=(x >= 28), color='coral', label='Hotter than 28°C')
plt.title('Midday Temperatures in July')
plt.xlabel('Temperature (°C)')
plt.legend()
plt.show()

# Filling the Gaps

The sensor only took a reading every two hours. What was the temperature at 9:30 in the morning? Estimating a value *between* known data points is called **interpolation**.

`CubicSpline` from **`scipy.interpolate`** draws a smooth curve that passes through every reading. The result, `curve`, works like a function: give it any hour and it returns the estimated temperature.

In [ ]:
from scipy import interpolate

curve = interpolate.CubicSpline(hours, temps)
print("Estimated temperature at 9:30am:", round(float(curve(9.5)), 1), "°C")

smooth_hours = np.linspace(0, 24, 200)

plt.figure(figsize=(10, 5))
plt.plot(smooth_hours, curve(smooth_hours), color='coral', label='Cubic spline')
plt.plot(hours, temps, 'o', color='steelblue', label='Sensor readings')
plt.title('Interpolating Between Readings')
plt.xlabel('Hour of the day')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Fitting a Model

A spline follows the data exactly, but it doesn't *explain* anything. Scientists often start with a **model** instead: a formula they believe describes what is happening, with a few unknown numbers in it.

Temperature rises and falls once a day, so a wave is a sensible model. Ours has three unknowns: the day's `average`, the `swing` above and below it, and the `peak_hour` when it is warmest.

`curve_fit` from **`scipy.optimize`** tries different values for the unknowns until the model sits as close to the data as possible. The `p0` list is our starting guess.

In [ ]:
from scipy import optimize

def daily_cycle(t, average, swing, peak_hour):
  return average + swing * np.cos(2 * np.pi * (t - peak_hour) / 24)

best, _ = optimize.curve_fit(daily_cycle, hours, temps, p0=[15, 5, 12])
average, swing, peak_hour = best

print("Average:", round(average, 1), "°C")
print("Swing:", round(swing, 1), "°C")
print("Peak hour:", round(peak_hour, 1))

plt.figure(figsize=(10, 5))
plt.plot(smooth_hours, daily_cycle(smooth_hours, *best), color='seagreen', label='Fitted model')
plt.plot(hours, temps, 'o', color='steelblue', label='Sensor readings')
plt.title('Fitting a Daily Cycle')
plt.xlabel('Hour of the day')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Finding the Best Value

**Optimization** means finding the input that makes a function as small (or as large) as possible. When exactly was the warmest moment of the day? The highest *reading* was at hour 14, but the true peak probably fell between two readings.

`minimize_scalar` searches for the lowest point of a function. SciPy only *minimizes*, so to find a maximum we flip the curve upside down by returning the negative temperature. The highest point becomes the lowest.

In [ ]:
def negative_temp(t):
  return -curve(t)

result = optimize.minimize_scalar(negative_temp, bounds=(0, 24))
peak = result.x

print("Warmest moment: hour", round(peak, 2))
print("Temperature then:", round(float(curve(peak)), 1), "°C")

## Solving an Equation

A closely related question: at what time did the temperature first reach 18°C? That means solving `curve(t) = 18`, which is the same as finding where `curve(t) - 18` equals zero.

`brentq` finds where a function crosses zero. You give it two inputs with the answer somewhere between them. Here we search between hour 6 and hour 14, while the morning is warming up.

In [ ]:
def above_18(t):
  return curve(t) - 18

t = optimize.brentq(above_18, 6, 14)
hour = int(t)
minute = int((t - hour) * 60)

print(f"The temperature reached 18°C at {hour}:{minute:02d}")

# Area Under a Curve

Adding up a quantity that changes continuously is called **integration**, and it means measuring the area under a curve. **`scipy.integrate`** does this with `quad`, which takes a function and the start and end of the range.

One use is a fairer daily average. Dividing the area under the temperature curve by 24 hours gives the average over *every moment* of the day, not just the 13 moments the sensor happened to record.

`quad` returns two numbers: the area, and an estimate of how far off it could be.

In [ ]:
from scipy import integrate

area, error = integrate.quad(curve, 0, 24)

print("Area under the curve:", round(area, 1))
print("Possible error:", error)
print("Average over the whole day:", round(area / 24, 2), "°C")
print("Average of the 13 readings:", round(temps.mean(), 2), "°C")

# Smoothing Noisy Data

Real sensors are never perfect. Every measurement includes a little random error, called **noise**. Let's imagine a cheaper sensor that reports every 10 minutes but jitters by a degree or so each time.

The **`scipy.signal`** submodule has tools for cleaning data like this. A **Savitzky-Golay filter** slides a window along the data and replaces each point with a smoothed value based on its neighbors.

Drag the slider to change the window size. A small window barely smooths at all. A large window is very smooth, but can flatten the real peak.

In [ ]:
from scipy import signal

WINDOW = 31 #@param {type:"slider", min:5, max:101, step:2}

rng = np.random.default_rng(7)
minutes = np.linspace(0, 24, 145)
noisy = curve(minutes) + rng.normal(0, 0.8, minutes.size)

smoothed = signal.savgol_filter(noisy, WINDOW, 3)

plt.figure(figsize=(10, 5))
plt.plot(minutes, noisy, '.', color='lightgray', label='Noisy sensor')
plt.plot(minutes, smoothed, color='crimson', linewidth=2, label=f'Smoothed (window {WINDOW})')
plt.title('Smoothing a Noisy Signal')
plt.xlabel('Hour of the day')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Check Your Understanding